In [24]:
!pip install groq

In [25]:
!pip install -q mcp groq

In [26]:
import os, getpass
os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key: ")
print("Key set!")

Groq API key: ··········
Key set!


In [27]:
%%writefile nutrition_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Nutrition Server", log_level="ERROR")

nutrition_data = {
    "chicken breast": {"calories": 165, "protein": 31},
    "banana": {"calories": 89, "protein": 1.1},
    "egg": {"calories": 155, "protein": 13},
    "oats": {"calories": 389, "protein": 16.9},
}

@mcp.tool()
def get_calories(food_item: str) -> str:
    """Get calorie information per 100g of a food item."""
    food = food_item.lower().strip()
    if food in nutrition_data:
        return f"{nutrition_data[food]['calories']} kcal"
    return f"No data available for {food_item}"

@mcp.tool()
def get_protein(food_item: str) -> str:
    """Get protein information per 100g of a food item."""
    food = food_item.lower().strip()
    if food in nutrition_data:
        return f"{nutrition_data[food]['protein']}g protein"
    return f"No data available for {food_item}"

if __name__ == "_main_":
    mcp.run(transport="stdio")

Overwriting nutrition_server.py


In [28]:
%%writefile nutrition_client.py
import os, sys, json, asyncio
from groq import AsyncGroq
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

MODEL = "openai/gpt-oss-20b"
SYSTEM = ("You are a Nutrition AI Agent. Always use the available tools for "
          "calorie or protein questions. Values are per 100g. Reply in one short "
          "sentence like: '100g of chicken breast contains approximately 165 calories.'")


async def ask_agent(session, groq_client, groq_tools, question):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    while True:
        resp = await groq_client.chat.completions.create(
            model=MODEL, messages=messages,
            tools=groq_tools, tool_choice="auto", temperature=0.2,
        )
        msg = resp.choices[0].message
        if not msg.tool_calls:
            return msg.content or ""

        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [{
                "id": tc.id, "type": "function",
                "function": {"name": tc.function.name,
                             "arguments": tc.function.arguments},
            } for tc in msg.tool_calls],
        })

        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            print(f"AI Agent: Uses the {tc.function.name} MCP tool.", flush=True)
            result = await session.call_tool(tc.function.name, args)
            text = "".join(c.text for c in result.content if c.type == "text")
            print(f"MCP Server: Returns {text}.", flush=True)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": text})


async def main():
    params = StdioServerParameters(command=sys.executable,
                                   args=["-u", "nutrition_server.py"])

    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            print("MCP Server Connected Successfully!\n", flush=True)

            listed = await asyncio.wait_for(session.list_tools(), timeout=15)
            print("Available Tools:", flush=True)
            for t in listed.tools:
                print(f"  - {t.name}: {t.description}", flush=True)
            print(f"Total Tools: {len(listed.tools)}\n", flush=True)

            groq_tools = [{
                "type": "function",
                "function": {"name": t.name,
                             "description": t.description or "",
                             "parameters": t.inputSchema},
            } for t in listed.tools]

            groq_client = AsyncGroq(api_key=os.environ["GROQ_API_KEY"])

            questions = [
                "How many calories are in 100g of chicken breast?",
                "How much protein is in 100g of egg?",
                "How many calories are in 100g of oats?",
            ]
            for q in questions:
                print("-" * 60, flush=True)
                print("User:", q, flush=True)
                answer = await ask_agent(session, groq_client, groq_tools, q)
                print("AI Agent:", answer, "\n", flush=True)


if __name__ == "_main_":
    asyncio.run(main())

Overwriting nutrition_client.py


In [31]:
!python nutrition_client.py